In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import glob


def load_experiment_results(path):
    import re
    model_slug = re.match(r"^(.*/)?elicit-beliefs-(.*)\.pt$", path).group(2)
    data = torch.load(path, weights_only=False)
    logits = data["logits"]
    top_other_logit = data["top_other_logit"]
    top_other_id = data["top_other_id"]
    top_other_token = data["top_other_token"]
    return model_slug, logits, top_other_logit, top_other_id, top_other_token


def compute_metrics(p_affirm, logits, top_other_logit):
    mean_p_affirm = p_affirm.mean(dim=1)
    certainty = np.array(torch.vstack((mean_p_affirm, 1 - mean_p_affirm)).max(axis=0))

    stdev_p_affirm = p_affirm.std(dim=1)
    stability = np.array(1 - 2 * stdev_p_affirm)

    # TODO: Leakage is computed is the sum of all other tokens, excluding
    # Yes, No, True, False, which isn't quite right since we should use either
    # (Yes, No) or (True, False) as viable tokens and not all 4 of them. That
    # depends on which prompt template was used though so, for now, this
    # calculation is good enough.
    all_logits = torch.hstack((logits, top_other_logit[:, None]))
    all_probs = all_logits.softmax(axis=1)
    leakage = np.array(all_probs[:, 4:].sum(axis=1))

    aggregate_function = np.mean
    
    return {
        'certainty': aggregate_function(certainty),
        'stability': aggregate_function(stability),
        'leakage': aggregate_function(leakage),
    }

In [2]:
from supporting_code import WORDS, YES, NO, TRUE, FALSE, PROMPT_TEMPLATES, logits_to_affirm_prob, present_results

In [3]:
path = "results/elicit-beliefs-gemma-2-27b-it.pt"
model_slug, logits, top_other_logit, top_other_id, top_other_token = load_experiment_results(path)

In [10]:
logits.isnan().sum()

tensor(1312608)

In [13]:
logits.shape

torch.Size([109384, 12])

In [14]:
109384*12

1312608